In [71]:
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd

# Allow running from either the project root or the test folder.
ROOT = Path.cwd()
ROOT = ROOT.parent

In [72]:
method = "test_bootstrap_nn"
# Variables
TEST_SIZE = 250
# qualified_gagne_1, qualified_gagne_2, qualified_gagne_3
QUALIFICATION_COLUMN = "qualified_gagne_1"

NEED_COLUMN = "gagne_sum_t"
GROUP_COLUMN = "race"
RASHOMON_EPSILON = 0.01
ALLOCATION_DELTA = 0.1
NUM_MODELS = 100
ITERATIONS_PER_SPLIT = 10

In [73]:
fp_raw = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
fp_predictions = ROOT / f"data/predictions/{method}_{QUALIFICATION_COLUMN}_with_expected_needs.csv"
df_raw = pd.read_csv(fp_raw)
df = pd.read_csv(fp_predictions)

In [74]:
df = df.merge(
    df_raw[["person_id", GROUP_COLUMN]],
    on="person_id",
    how="left",
    validate="many_to_one",
)

In [78]:
df_diversity = pd.read_csv(ROOT / "data/results/diversity_all.csv")

In [75]:
all_results = []

for SELECTION_RATE in [0.1, 0.25, 0.5]:
    fp_input_opt = ROOT / f"data/results/allocations/opt_exp_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
    fp_input_lda_tru = ROOT / f"data/results/allocations/lda_tru_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
    fp_input_lda_exp = ROOT / f"data/results/allocations/lda_exp_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
    fp_input_people_tru = ROOT / f"data/results/allocations/people_tru_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
    fp_input_people_exp = ROOT / f"data/results/allocations/people_exp_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"

    df_opt = pd.read_csv(fp_input_opt, converters={"selected": literal_eval, "models": literal_eval})
    df_lda_tru = pd.read_csv(fp_input_lda_tru, converters={"selected": literal_eval, "models": literal_eval})
    df_lda_exp = pd.read_csv(fp_input_lda_exp, converters={"selected": literal_eval, "models": literal_eval})
    # df_people_tru = pd.read_csv(fp_input_people_tru, converters={"people": literal_eval})
    df_people_exp = pd.read_csv(fp_input_people_exp, converters={"people": literal_eval})

    k = int(SELECTION_RATE*TEST_SIZE)

    for _, cohort_info in df_people_exp.iterrows():
        seed = cohort_info["seed"]
        iteration = cohort_info["iteration"]

        split_df = df.loc[(df["seed"] == seed) & (df["person_id"] >= 0)]
        split_df = split_df.set_index("person_id")
        cohort = split_df.loc[cohort_info["people"]]

        for algorithm, allocations in [("OPT", df_opt), ("LDA-tru", df_lda_tru), ("LDA-exp", df_lda_exp)]:
        # for algorithm, allocations in [("OPT", df_opt), ("LDA-exp", df_lda_exp)]:
            cohort_allocations = allocations.loc[(allocations["seed"] == seed) & (allocations["iteration"] == iteration)]

            # Count selected sets, not model rows or allocation_idx values.
            # Sorting makes [person_a, person_b] and [person_b, person_a] identical.
            unique_allocations = {
                tuple(sorted(selected_ids)) for selected_ids in cohort_allocations["selected"]
            }
            model_names = {
                model for models in cohort_allocations["models"] for model in models
            }
            num_unique_allocations = len(unique_allocations)
            num_models = len(model_names)
            unique_allocation_ratio = num_unique_allocations / num_models if num_models else np.nan

            for _, allocation in cohort_allocations.iterrows():
                selected = cohort.loc[allocation["selected"]]

                mean_0 = selected.loc[selected[GROUP_COLUMN] == 0, NEED_COLUMN].mean()
                mean_1 = selected.loc[selected[GROUP_COLUMN] == 1, NEED_COLUMN].mean()
                absolute_disparity = abs(mean_1 - mean_0)
                disparity_ratio = np.nan
                if pd.notna(mean_0) and pd.notna(mean_1) and mean_0 != 0 and mean_1 != 0:
                    disparity_ratio = max(mean_1 / mean_0, mean_0 / mean_1)

                for model in allocation["models"]:
                    all_results.append({
                        "method": method,
                        "algorithm": algorithm,
                        "qualification_column": QUALIFICATION_COLUMN,
                        "seed": seed,
                        "iteration": iteration,
                        "model": model,
                        "k": k,
                        "epsilon": ALLOCATION_DELTA,
                        "tolerance_type": "add",
                        "expected_utility": selected[model].mean(),
                        "realized_utility": selected["y"].mean(),
                        "absolute_disparity": absolute_disparity,
                        "disparity_ratio": disparity_ratio,
                        "num_models": num_models,
                        "num_unique_allocations": num_unique_allocations,
                        "unique_allocation_ratio": unique_allocation_ratio,
                    })

    # for _, cohort_info in df_people_tru.iterrows():
    #         seed = cohort_info["seed"]
    #         iteration = cohort_info["iteration"]
    
    #         split_df = df.loc[(df["seed"] == seed) & (df["person_id"] >= 0)]
    #         split_df = split_df.set_index("person_id")
    #         cohort = split_df.loc[cohort_info["people"]]
    
    #         for algorithm, allocations in [("LDA-tru", df_lda_tru)]:
    #             cohort_allocations = allocations.loc[(allocations["seed"] == seed) & (allocations["iteration"] == iteration)]
    
    #             # Count selected sets, not model rows or allocation_idx values.
    #             # Sorting makes [person_a, person_b] and [person_b, person_a] identical.
    #             unique_allocations = {
    #                 tuple(sorted(selected_ids)) for selected_ids in cohort_allocations["selected"]
    #             }
    #             model_names = {
    #                 model for models in cohort_allocations["models"] for model in models
    #             }
    #             num_unique_allocations = len(unique_allocations)
    #             num_models = len(model_names)
    #             unique_allocation_ratio = num_unique_allocations / num_models if num_models else np.nan
    
    #             for _, allocation in cohort_allocations.iterrows():
    #                 selected = cohort.loc[allocation["selected"]]
    
    #                 mean_0 = selected.loc[selected[GROUP_COLUMN] == 0, NEED_COLUMN].mean()
    #                 mean_1 = selected.loc[selected[GROUP_COLUMN] == 1, NEED_COLUMN].mean()
    #                 absolute_disparity = abs(mean_1 - mean_0)
    #                 disparity_ratio = np.nan
    #                 if pd.notna(mean_0) and pd.notna(mean_1) and mean_0 != 0 and mean_1 != 0:
    #                     disparity_ratio = max(mean_1 / mean_0, mean_0 / mean_1)
    
    #                 for model in allocation["models"]:
    #                     all_results.append({
    #                         "method": method,
    #                         "algorithm": algorithm,
    #                         "qualification_column": QUALIFICATION_COLUMN,
    #                         "seed": seed,
    #                         "iteration": iteration,
    #                         "model": model,
    #                         "k": k,
    #                         "epsilon": ALLOCATION_DELTA,
    #                         "tolerance_type": "add",
    #                         "expected_utility": selected[model].mean(),
    #                         "realized_utility": selected["y"].mean(),
    #                         "absolute_disparity": absolute_disparity,
    #                         "disparity_ratio": disparity_ratio,
    #                         "num_models": num_models,
    #                         "num_unique_allocations": num_unique_allocations,
    #                         "unique_allocation_ratio": unique_allocation_ratio,
    #                     })

        

results = pd.DataFrame(all_results)
results

,method,algorithm,qualification_column,seed,iteration,model,k,epsilon,tolerance_type,expected_utility,realized_utility,absolute_disparity,disparity_ratio,num_models,num_unique_allocations,unique_allocation_ratio
0,test_bootstrap_nn,OPT,qualified_gagne_1,0,0,m_1,25,0.1,add,0.980055,1.000,NaN,NaN,100,100,1.00
1,test_bootstrap_nn,OPT,qualified_gagne_1,0,0,m_2,25,0.1,add,0.980080,1.000,NaN,NaN,100,100,1.00
2,test_bootstrap_nn,OPT,qualified_gagne_1,0,0,m_3,25,0.1,add,0.990442,1.000,NaN,NaN,100,100,1.00
3,test_bootstrap_nn,OPT,qualified_gagne_1,0,0,m_4,25,0.1,add,0.981349,1.000,NaN,NaN,100,100,1.00
4,test_bootstrap_nn,OPT,qualified_gagne_1,0,0,m_5,25,0.1,add,0.987068,0.960,NaN,NaN,100,100,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89995,test_bootstrap_nn,LDA-exp,qualified_gagne_1,9,9,m_95,125,0.1,add,0.906005,0.912,0.304878,1.138889,100,93,0.93
89996,test_bootstrap_nn,LDA-exp,qualified_gagne_1,9,9,m_96,125,0.1,add,0.907244,0.904,0.345528,1.160377,100,93,0.93
89997,test_bootstrap_nn,LDA-exp,qualified_gagne_1,9,9,m_97,125,0.1,add,0.896130,0.912,0.272358,1.122263,100,93,0.93
89998,test_bootstrap_nn,LDA-exp,qualified_gagne_1,9,9,m_98,125,0.1,add,0.898776,0.912,0.296748,1.134686,100,93,0.93


In [94]:
columns = ["expected_utility", "realized_utility", "absolute_disparity", "disparity_ratio", "num_unique_allocations", "unique_allocation_ratio", "num_models"]
cols = ["pairwise_consistency", "share_chance_ge_10pct", "systemic_exclusion_all", "systemic_exclusion_qualified"]
grouped = results.groupby(["k", "algorithm"])[columns].agg(["mean", "std"])
summary = pd.DataFrame(index=grouped.index)

for col in columns:
    summary[col] = grouped.apply(lambda row: f"{row[(col, 'mean')]:.2f} \u00b1 {row[(col, 'std')]:.2f}", axis=1)
# summary

In [93]:
df_diversity_grouped = df_diversity.drop(columns=["tag", "seed"]).groupby(["k", "algorithm"]).agg(["mean", "std"])
summary_diversity = pd.DataFrame(index=df_diversity_grouped.index)
for col in cols:
    summary_diversity[col] = df_diversity_grouped.apply(lambda row: f"{row[(col, 'mean')]:.2f} \u00b1 {row[(col, 'std')]:.2f}", axis=1)

|Metric| Description|
|---|---|
|Pairwise Consistency|Share of people given the same outcome by two random allocations. The paper's homogenization measure|
|Share Change ge 10pct| Fraction of the individuals selected by at least 10% of models|
|Systemic Exclusion All| Fraction of the individuals selected by no model|
|Systemic Exlusion Qualified| Same, among qualified people only — the quantity the paper reports|

In [96]:
pd.merge(summary, summary_diversity, left_index=True, right_index=True).drop(columns=["num_models"])

expected_utility realized_utility absolute_disparity  \
k   algorithm                                                        
25  LDA-exp        0.98 ± 0.01      0.98 ± 0.03        0.97 ± 0.91   
    LDA-tru        0.98 ± 0.01      0.98 ± 0.04        0.00 ± 0.00   
    OPT            0.99 ± 0.00      0.98 ± 0.03        1.49 ± 1.15   
62  LDA-exp        0.96 ± 0.01      0.97 ± 0.03        0.57 ± 0.47   
    LDA-tru        0.96 ± 0.01      0.97 ± 0.03        0.00 ± 0.00   
    OPT            0.97 ± 0.01      0.97 ± 0.02        0.99 ± 0.69   
125 LDA-exp        0.92 ± 0.02      0.93 ± 0.03        0.47 ± 0.58   
    LDA-tru        0.92 ± 0.02      0.93 ± 0.03        0.04 ± 0.42   
    OPT            0.92 ± 0.01      0.93 ± 0.03        0.96 ± 0.77   

              disparity_ratio num_unique_allocations unique_allocation_ratio  \
k   algorithm                                                                  
25  LDA-exp       1.34 ± 0.71           95.88 ± 5.12             0.96 ± 0.05   
    LDA-tru       1.00 ± 0.00           96.40 ± 4.79             0.96 ± 0.05   
    OPT           1.44 ± 0.61           98.68 ± 2.82             0.99 ± 0.03   
62  LDA-exp       1.17 ± 0.17           99.44 ± 1.40             0.99 ± 0.01   
    LDA-tru       1.00 ± 0.00           99.54 ± 1.40             1.00 ± 0.01   
    OPT           1.31 ± 0.24           99.91 ± 0.45             1.00 ± 0.00   
125 LDA-exp       1.21 ± 0.29          94.69 ± 12.03             0.95 ± 0.12   
    LDA-tru       1.01 ± 0.11          92.02 ± 15.44             0.92 ± 0.15   
    OPT           1.43 ± 0.42           98.36 ± 4.04             0.98 ± 0.04   

              pairwise_consistency share_chance_ge_10pct  \
k   algorithm                                              
25  LDA-exp            0.95 ± 0.01           0.16 ± 0.01   
    LDA-tru            0.95 ± 0.01           0.16 ± 0.01   
    OPT                0.96 ± 0.01           0.15 ± 0.01   
62  LDA-exp            0.93 ± 0.01           0.33 ± 0.02   
    LDA-tru            0.93 ± 0.01           0.32 ± 0.01   
    OPT                0.94 ± 0.01           0.32 ± 0.01   
125 LDA-exp            0.95 ± 0.02           0.55 ± 0.02   
    LDA-tru            0.95 ± 0.02           0.55 ± 0.02   
    OPT                0.95 ± 0.01           0.55 ± 0.01   

              systemic_exclusion_all systemic_exclusion_qualified  
k   algorithm                                                      
25  LDA-exp              0.77 ± 0.03                  0.64 ± 0.04  
    LDA-tru              0.78 ± 0.02                  0.65 ± 0.03  
    OPT                  0.79 ± 0.02                  0.67 ± 0.03  
62  LDA-exp              0.61 ± 0.02                  0.38 ± 0.04  
    LDA-tru              0.61 ± 0.02                  0.38 ± 0.05  
    OPT                  0.61 ± 0.02                  0.39 ± 0.04  
125 LDA-exp              0.42 ± 0.03                  0.13 ± 0.04  
    LDA-tru              0.42 ± 0.03                  0.13 ± 0.03  
    OPT                  0.42 ± 0.02                  0.13 ± 0.04